requires GPU compute

In [ ]:
#!pip install transformers==4.46.0

downgrade to <5.0.0 transformers, to avoid `special_image_mask.unsqueeze(-1)` failing with error: `'bool' object has no attribute 'unsqueeze'`
and ensure correct, expected working


## Test - English caption generation for 5 images

In [ ]:
from transformers import Blip2ForConditionalGeneration

model_id = "Gregor/mblip-mt0-xl"

model = Blip2ForConditionalGeneration.from_pretrained(model_id)

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

In [ ]:
import requests
from PIL import Image
from io import BytesIO

urls = [
    "https://images.unsplash.com/photo-1518717758536-85ae29035b6d",
    "https://images.unsplash.com/photo-1503023345310-bd7c1de61c7d",
    "https://images.unsplash.com/photo-1500530855697-b586d89ba3ee",
    "https://images.unsplash.com/photo-1441974231531-c6227db76b6e",
    "https://images.unsplash.com/photo-1507149833265-60c372daea22"
]

images = []

for url in urls:
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(img)

print("Loaded", len(images), "images")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for i,img in enumerate(images):
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

In [ ]:
def run_mblip(image, prompt="<image> Describe the image.", max_new_tokens=50):
    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
      out = model.generate(**inputs, max_new_tokens=50)

    return processor.decode(out[0], skip_special_tokens=True)

In [ ]:
for i, img in enumerate(images):
    prompt1 = "<image> Describe the image in English."
    caption = run_mblip(img, prompt=prompt1)
    print(f"Image {i+1}")
    print("Caption:", caption)
    print("-" * 40)

## TODO - BLEU, CiDER, and mCLIP evaluation for <insert languauges> to reproduce paper results